# Lab: Egg Detection, Counting & Classification

**Duration:** ~50 minutes

**Objective:** Build a complete pipeline that (1) detects and counts eggs using YOLOv8, then (2) classifies egg mutations using color histogram features and a Random Forest.

## Pipeline Overview

| Part | Task | Method |
|------|------|--------|
| **Part 1** | Clone data & setup | Git + Ultralytics |
| **Part 2** | Train YOLOv8 for egg detection & counting | Object Detection (20 epochs) |
| **Part 3** | Download training outputs | Ultralytics graphs + model weights |
| **Part 4** | Run inference on classification images | YOLOv8 bounding boxes |
| **Part 5** | Crop eggs & remove background | Bounding box extraction |
| **Part 6** | Extract color histograms & plot | Feature engineering |
| **Part 7** | Train/Val/Test split | Stratified split |
| **Part 8** | Train Random Forest & evaluate | Acc, Precision, Recall, F1, Confusion Matrix |

---
## Part 1 — Setup & Clone Repository

In [ ]:
!pip install ultralytics -q

In [ ]:
# Clone the repository with datasets
!git clone https://github.com/DoreaLab/digag.git /content/digag 2>/dev/null || echo "Repository already cloned."

import os
import cv2
import shutil
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from ultralytics import YOLO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Paths
YOLO_DATA   = "/content/digag/Egg.v12-egg.yolov8"
DATA_DIR    = "/content/digag/Data_eggs"
CLASSES     = ["Mutu A", "Mutu B", "Mutu C"]
IMG_SIZE    = (256, 256)
SEED        = 42
N_BINS      = 64

print("=" * 50)
print("YOLO Detection Dataset:")
print(f"  Train: {len(os.listdir(os.path.join(YOLO_DATA, 'train/images')))} images")
print(f"  Valid: {len(os.listdir(os.path.join(YOLO_DATA, 'valid/images')))} images")
print(f"  Test:  {len(os.listdir(os.path.join(YOLO_DATA, 'test/images')))} images")
print("\nClassification Dataset:")
for cls in CLASSES:
    n = len(os.listdir(os.path.join(DATA_DIR, cls)))
    print(f"  {cls}: {n} images")
print("=" * 50)

### Visualize Detection Dataset Samples

In [ ]:
# Show a few training images with their bounding box annotations
train_img_dir = os.path.join(YOLO_DATA, 'train/images')
train_lbl_dir = os.path.join(YOLO_DATA, 'train/labels')
sample_files = sorted(os.listdir(train_img_dir))[:6]

yolo_classes = ['blue egg', 'red egg', 'white egg']
yolo_colors  = [(255, 0, 0), (0, 0, 255), (200, 200, 200)]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for idx, fname in enumerate(sample_files):
    ax = axes[idx // 3, idx % 3]
    img = cv2.imread(os.path.join(train_img_dir, fname))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # Draw YOLO annotations
    lbl_path = os.path.join(train_lbl_dir, os.path.splitext(fname)[0] + '.txt')
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(v) for v in parts[1:]]
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), yolo_colors[cls_id], 2)
                cv2.putText(img, yolo_classes[cls_id], (x1, y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, yolo_colors[cls_id], 2)

    ax.imshow(img)
    ax.axis('off')

fig.suptitle('YOLO Training Samples with Bounding Box Annotations', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## Part 2 — Train YOLOv8 for Egg Detection & Counting

We fine-tune a pre-trained YOLOv8n model on our egg dataset for **20 epochs**.

The model learns to detect and localize 3 types of eggs: `blue egg`, `red egg`, `white egg`.

In [ ]:
# Fix data.yaml paths for Colab
data_yaml = os.path.join(YOLO_DATA, 'data.yaml')
with open(data_yaml, 'r') as f:
    content = f.read()

# Replace relative paths with absolute paths
new_content = content.replace('../train/images', f'{YOLO_DATA}/train/images')
new_content = new_content.replace('../valid/images', f'{YOLO_DATA}/valid/images')
new_content = new_content.replace('../test/images', f'{YOLO_DATA}/test/images')

with open(data_yaml, 'w') as f:
    f.write(new_content)

print("Updated data.yaml:")
print(new_content)

In [ ]:
# Train YOLOv8n for 20 epochs
model = YOLO('yolov8n.pt')

results = model.train(
    data=data_yaml,
    epochs=20,
    imgsz=640,
    batch=16,
    name='egg_detector',
    project='/content/yolo_output',
    exist_ok=True,
    verbose=True
)

---
## Part 3 — Explore Training Results & Download Model

Ultralytics saves training curves, confusion matrix, and model weights automatically.

In [ ]:
# Display training curves
from IPython.display import Image, display

output_dir = '/content/yolo_output/egg_detector'

plots = ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
         'F1_curve.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png']

for plot_name in plots:
    plot_path = os.path.join(output_dir, plot_name)
    if os.path.exists(plot_path):
        print(f"\n{'='*50}")
        print(f"  {plot_name}")
        print(f"{'='*50}")
        display(Image(filename=plot_path, width=800))

In [ ]:
# Download the full training output (weights, graphs, logs)
zip_path = '/content/yolo_egg_detector_output.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', output_dir)

print(f"Training output zipped: {zip_path}")
print(f"Contents:")
for root, dirs, fls in os.walk(output_dir):
    level = root.replace(output_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for f in fls[:10]:
        print(f"{sub_indent}{f}")

print("\nDownloading...")
files.download(zip_path)

---
## Part 4 — Run YOLO Inference on Classification Images

Now we use the trained egg detector to find eggs in the `Data_eggs` images (Mutu A, B, C).

For each detected egg, we extract the bounding box region — this gives us a clean crop of each egg with the background removed.

In [ ]:
# Load the best trained model
best_model = YOLO(os.path.join(output_dir, 'weights/best.pt'))
print(f"Loaded model: {best_model.ckpt_path}")

In [ ]:
# Run inference on all Data_eggs images and crop detections
cropped_images = []  # list of (crop_bgr, class_name)
no_detection = 0

for cls in CLASSES:
    folder = os.path.join(DATA_DIR, cls)
    image_files = sorted(os.listdir(folder))
    print(f"\nProcessing {cls} ({len(image_files)} images)...", end=" ")
    cls_count = 0

    for fname in image_files:
        fpath = os.path.join(folder, fname)
        img = cv2.imread(fpath)
        if img is None:
            continue

        # Run YOLO inference
        preds = best_model.predict(img, conf=0.25, verbose=False)
        boxes = preds[0].boxes

        if len(boxes) == 0:
            # No detection — use full image as fallback
            cropped_images.append((img, cls))
            no_detection += 1
            cls_count += 1
            continue

        # Extract crops for each detected egg
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            # Clamp to image bounds
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
            crop = img[y1:y2, x1:x2]
            if crop.size > 0:
                cropped_images.append((crop, cls))
                cls_count += 1

    print(f"{cls_count} egg crops extracted.")

print(f"\nTotal crops: {len(cropped_images)}")
print(f"Images with no detection (used full image): {no_detection}")

---
## Part 5 — Visualize YOLO Detections & Cropped Eggs

Let's see how the detector performs on the classification images and what the crops look like.

In [ ]:
# Show detection + crop for 2 images per class
fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for row, cls in enumerate(CLASSES):
    folder = os.path.join(DATA_DIR, cls)
    sample_files = sorted(os.listdir(folder))[:2]

    for i, fname in enumerate(sample_files):
        fpath = os.path.join(folder, fname)
        img = cv2.imread(fpath)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        preds = best_model.predict(img, conf=0.25, verbose=False)
        boxes = preds[0].boxes

        # Draw detections
        img_det = img_rgb.copy()
        crop_rgb = None
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            conf = box.conf[0].cpu().numpy()
            cv2.rectangle(img_det, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(img_det, f'{conf:.2f}', (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
            if crop_rgb is None:
                x1c, y1c = max(0, x1), max(0, y1)
                x2c, y2c = min(img.shape[1], x2), min(img.shape[0], y2)
                crop_rgb = img_rgb[y1c:y2c, x1c:x2c]

        col = i * 2
        axes[row, col].imshow(img_det)
        axes[row, col].set_title(f"{cls} — Detection ({len(boxes)} eggs)", fontsize=11)
        axes[row, col].axis('off')

        if crop_rgb is not None:
            axes[row, col+1].imshow(crop_rgb)
            axes[row, col+1].set_title("Cropped Egg (no background)", fontsize=11)
        else:
            axes[row, col+1].text(0.5, 0.5, 'No detection', ha='center', va='center')
        axes[row, col+1].axis('off')

fig.suptitle('YOLO Detection on Classification Images + Bounding Box Crops', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## Part 6 — Extract Color Histograms from Cropped Eggs

Now we extract normalized RGB histograms from the **YOLO-cropped egg regions** (not the full image).

This is better than using the full image because:
- The background is removed by the bounding box
- Only egg pixels contribute to the color distribution

Feature vector: `N_BINS x 3 channels` = 192 features per egg.

In [ ]:
def extract_histogram(img_bgr, n_bins=N_BINS):
    """Extract normalized RGB histogram from an image region."""
    img_resized = cv2.resize(img_bgr, IMG_SIZE)
    hists = []
    for ch in range(3):  # B, G, R
        h = cv2.calcHist([img_resized], [ch], None, [n_bins], [0, 256])
        h = h.flatten()
        h = h / (h.sum() + 1e-8)  # normalize to probability distribution
        hists.append(h)
    return np.concatenate(hists)  # shape: (n_bins * 3,)

# Extract features from all cropped eggs
features = []
labels = []
class_to_idx = {cls: i for i, cls in enumerate(CLASSES)}

for crop_bgr, cls_name in cropped_images:
    hist = extract_histogram(crop_bgr)
    features.append(hist)
    labels.append(class_to_idx[cls_name])

X = np.array(features)
y = np.array(labels)

print(f"Feature matrix shape: {X.shape}  (samples x features)")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {dict(zip(CLASSES, np.bincount(y)))}")

### Plot Average Color Histograms per Class

In [ ]:
channel_names = ["Blue", "Green", "Red"]
channel_colors = ["blue", "green", "red"]

# Per-channel comparison across classes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ch_idx in range(3):
    ax = axes[ch_idx]
    start = ch_idx * N_BINS
    end = start + N_BINS
    bins_x = np.linspace(0, 255, N_BINS)
    for class_idx, cls in enumerate(CLASSES):
        mask_cls = y == class_idx
        mean_hist = X[mask_cls, start:end].mean(axis=0)
        ax.plot(bins_x, mean_hist, label=cls, linewidth=2)
    ax.set_title(f"{channel_names[ch_idx]} Channel", fontsize=14)
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Normalized Frequency")
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle("Average Color Histogram per Class (YOLO-Cropped Eggs)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# RGB overlay per class
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bins_x = np.linspace(0, 255, N_BINS)

for class_idx, cls in enumerate(CLASSES):
    ax = axes[class_idx]
    mask_cls = y == class_idx
    for ch_idx, (ch_name, ch_color) in enumerate(zip(channel_names, channel_colors)):
        start = ch_idx * N_BINS
        end = start + N_BINS
        mean_hist = X[mask_cls, start:end].mean(axis=0)
        ax.fill_between(bins_x, mean_hist, alpha=0.3, color=ch_color)
        ax.plot(bins_x, mean_hist, color=ch_color, label=ch_name, linewidth=1.5)
    ax.set_title(cls, fontsize=14)
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Normalized Frequency")
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle("RGB Distribution per Class (YOLO-Cropped)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 7 — Train / Validation / Test Split

Stratified split to maintain class balance:
- **Train**: 70%
- **Validation**: 15%
- **Test**: 15%

In [ ]:
# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
# Second split: 50/50 of temp -> 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train:      {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Class distribution per split
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, labels_split) in zip(axes, [("Train", y_train), ("Validation", y_val), ("Test", y_test)]):
    counts = np.bincount(labels_split, minlength=3)
    ax.bar(CLASSES, counts, color=["#4C72B0", "#55A868", "#C44E52"])
    ax.set_title(f"{name} (n={len(labels_split)})")
    ax.set_ylabel("Count")
    for i, c in enumerate(counts):
        ax.text(i, c + 1, str(c), ha="center", fontweight="bold")
fig.suptitle("Class Distribution per Split", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 8 — Train Random Forest & Evaluate

**Input:** Normalized color histogram (192 features = 64 bins x 3 channels) from YOLO-cropped eggs.

**Output:** Egg class (Mutu A, B, or C).

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    random_state=SEED,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Validation performance
y_val_pred = rf.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=CLASSES))

### Feature Importance

In [ ]:
importances = rf.feature_importances_
fig, ax = plt.subplots(figsize=(14, 4))
colors = ["blue"] * N_BINS + ["green"] * N_BINS + ["red"] * N_BINS
ax.bar(range(len(importances)), importances, color=colors, alpha=0.7, width=1.0)
ax.set_xlabel("Feature Index (B: 0-63 | G: 64-127 | R: 128-191)")
ax.set_ylabel("Importance")
ax.set_title("Random Forest Feature Importances by Histogram Bin")
ax.axvline(x=N_BINS - 0.5, color="black", linestyle="--", alpha=0.5)
ax.axvline(x=2 * N_BINS - 0.5, color="black", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Test Set Evaluation

In [ ]:
y_test_pred = rf.predict(X_test)

acc  = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred, average="weighted")
rec  = recall_score(y_test, y_test_pred, average="weighted")
f1   = f1_score(y_test, y_test_pred, average="weighted")

print("=" * 45)
print("       TEST SET METRICS")
print("=" * 45)
print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}  (weighted)")
print(f"  Recall:    {rec:.4f}  (weighted)")
print(f"  F1-Score:  {f1:.4f}  (weighted)")
print("=" * 45)
print(f"\nPer-class Report:\n")
print(classification_report(y_test, y_test_pred, target_names=CLASSES))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp1.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix (Counts)")

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASSES)
disp2.plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("Confusion Matrix (Normalized)")

fig.suptitle("Test Set Confusion Matrix", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

| Part | What we did |
|------|-------------|
| 1 | Cloned repo with detection + classification datasets |
| 2 | Trained YOLOv8n for 20 epochs on egg detection (blue/red/white) |
| 3 | Explored Ultralytics training curves & downloaded model |
| 4 | Ran YOLO inference on Mutu A/B/C images to detect & crop eggs |
| 5 | Visualized detections and bounding box crops |
| 6 | Extracted normalized RGB histograms (192 features) from crops |
| 7 | Split data: 70% Train / 15% Val / 15% Test (stratified) |
| 8 | Trained Random Forest, evaluated with Acc, Precision, Recall, F1, Confusion Matrix |

### Key Takeaways
- **YOLOv8** provides automatic egg localization, eliminating manual cropping or threshold-based segmentation.
- **Bounding box crops** remove background noise more reliably than fixed thresholds.
- **Color histograms** from cropped regions capture the true egg color distribution.
- **Random Forest** on histogram features is a simple yet effective classifier for egg mutation types.
- The pipeline combines deep learning (detection) with classical ML (classification) — a powerful and interpretable approach.